# Bank Marketing Term Deposit Prediction
Predict whether a client will subscribe to a term deposit from phone marketing campaign data.

## Project Overview
Binary classification on the Bank Marketing dataset. Predict term deposit subscription from client demographics, campaign contacts, and economic indicators.

## Learning Objectives
- Handle imbalanced marketing response data
- Work with mixed numeric/categorical bank features
- Understand campaign optimization through ML

## Problem Statement
Given client information and campaign contact details, predict whether the client will subscribe to a term deposit (yes/no).

## Why This Project Matters
Direct marketing campaigns are expensive. Predicting likely subscribers helps banks target resources effectively and improve conversion rates.

## Dataset Overview
| Feature | Value |
|---|---|
| **Source** | Kaggle: henriqueyamahata/bank-marketing |
| **Target** | y (yes/no — subscribed to term deposit) |
| **Rows** | ~41,000 |
| **Features** | age, job, marital, education, contact, campaign, etc. |

## Dataset Source & License
UCI ML Repository via Kaggle. Bank Marketing dataset. CC BY 4.0.

## Environment Setup

In [1]:
import subprocess, sys
for pkg in ['catboost','lightgbm','xgboost','flaml','lazypredict']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Imports

In [2]:
import os, json, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
TEST_SIZE = 0.2
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
os.makedirs(SAVE_DIR, exist_ok=True)

## Configuration

In [3]:
MAX_ROWS = 50000

## Dataset Loading

In [4]:
import kagglehub
path = kagglehub.dataset_download('henriqueyamahata/bank-marketing')
import glob
csvs = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
print('Files:', [os.path.basename(f) for f in csvs])
# Use the largest CSV
csvs.sort(key=lambda x: os.path.getsize(x), reverse=True)
df = pd.read_csv(csvs[0], sep=';' if 'bank' in csvs[0].lower() else ',')
# Try semicolon separator if columns look wrong
if len(df.columns) <= 2:
    df = pd.read_csv(csvs[0], sep=';')
df.columns = df.columns.str.strip()
if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED)
print(f'Shape: {df.shape}')
print(df.columns.tolist())
df.head()

  0%|                                               | 0.00/393k [00:00<?, ?B/s]

100%|████████████████████████████████████████| 393k/393k [00:01<00:00, 306kB/s]

100%|████████████████████████████████████████| 393k/393k [00:01<00:00, 306kB/s]

Extracting files...


Files: ['bank-additional-full.csv']
Shape: (41188, 21)
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## Data Validation

In [5]:
print('Missing:', df.isnull().sum().sum())
print(f'Duplicates: {df.duplicated().sum()}')

# Find target
y_cols = [c for c in df.columns if c.lower() in ('y', 'deposit', 'subscribed')]
TARGET = y_cols[0] if y_cols else df.columns[-1]
print(f'\nTarget: {TARGET}')
print(df[TARGET].value_counts())

Missing: 0
Duplicates: 12

Target: y
y
no     36548
yes     4640
Name: count, dtype: int64


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df[TARGET].value_counts().plot.bar(ax=axes[0,0], color=['#3498db','#e74c3c'], edgecolor='black')
axes[0,0].set_title('Term Deposit Subscription')

if 'age' in df.columns:
    for label in df[TARGET].unique()[:2]:
        axes[0,1].hist(df[df[TARGET]==label]['age'], bins=30, alpha=0.5, label=str(label))
    axes[0,1].legend()
    axes[0,1].set_title('Age by Subscription')

if 'job' in df.columns:
    df['job'].value_counts().head(10).plot.bar(ax=axes[1,0], edgecolor='black')
    axes[1,0].set_title('Job Distribution')
    axes[1,0].tick_params(axis='x', rotation=45)

if 'duration' in df.columns:
    axes[1,1].hist(df['duration'].clip(upper=1000), bins=50, edgecolor='black')
    axes[1,1].set_title('Call Duration (clipped)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## Target Analysis

In [7]:
print(df[TARGET].value_counts())
print(f'\nPositive rate: {(df[TARGET].isin(["yes",1,"1"])).mean():.2%}')

y
no     36548
yes     4640
Name: count, dtype: int64

Positive rate: 11.27%


## Train/Test Split

In [8]:
# Encode target
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df[TARGET] = le.fit_transform(df[TARGET])
print(f'Classes: {le.classes_}')

# Encode categoricals
cat_cols = df.select_dtypes(include='object').columns.tolist()
for c in cat_cols:
    df[c] = df[c].astype('category').cat.codes

df = df.fillna(df.median(numeric_only=True))
df = df.replace([np.inf, -np.inf], np.nan).fillna(0)

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Classes: ['no' 'yes']
Train: (32950, 20), Test: (8238, 20)


## Preprocessing
Label-encoded target and categoricals. Filled missing values.

## Feature Engineering
Using raw features. Note: 'duration' is known after call — would be leakage in production.

## Baseline Model

In [9]:
bl = LogisticRegression(max_iter=1000, random_state=SEED)
bl.fit(X_train, y_train)
bl_pred = bl.predict(X_test)
print(f'Baseline LR: Acc={accuracy_score(y_test, bl_pred):.4f}  F1={f1_score(y_test, bl_pred):.4f}')

Baseline LR: Acc=0.9126  F1=0.5115


## LazyPredict Benchmark

In [10]:
try:
    from lazypredict.Supervised import LazyClassifier
    n_lp = min(5000, len(X_train))
    lc = LazyClassifier(verbose=0, ignore_warnings=True)
    lp_models, _ = lc.fit(X_train.head(n_lp), X_test.head(min(1000, len(X_test))),
                          y_train.head(n_lp), y_test.head(min(1000, len(y_test))))
    print(lp_models.head(15))
except Exception as e:
    print(f"LazyPredict skipped: {e}")

                               Accuracy  Balanced Accuracy   ROC AUC  \
Model                                                                  
Perceptron                        0.887           0.808673  0.903366   
NearestCentroid                   0.775           0.766914  0.888872   
QuadraticDiscriminantAnalysis     0.874           0.754051  0.907239   
BernoulliNB                       0.823           0.750546  0.830459   
GaussianNB                        0.854           0.749969  0.871560   
XGBClassifier                     0.907           0.747341  0.934919   
PassiveAggressiveClassifier       0.881           0.743488  0.882792   
LGBMClassifier                    0.912           0.742911  0.939383   
CatBoostClassifier                0.914           0.740411  0.942006   
DecisionTreeClassifier            0.895           0.736897  0.736897   
LinearDiscriminantAnalysis        0.913           0.725307  0.940700   
RandomForestClassifier            0.910           0.723605  0.94

## FLAML AutoML

In [11]:
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train, y_train, task='classification', time_budget=60, seed=SEED, verbose=0)
    flaml_pred = automl.predict(X_test)
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  Acc={accuracy_score(y_test, flaml_pred):.4f}  F1={f1_score(y_test, flaml_pred, average='weighted'):.4f}")
except Exception as e:
    print(f"FLAML skipped: {e}")

FLAML skipped: XGBClassifier.fit() got an unexpected keyword argument 'callbacks'


## Additional Models: CatBoost, LightGBM, XGBoost

In [12]:
models = {}
cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0)
cb.fit(X_train, y_train)
models['CatBoost'] = cb

lgb = LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1)
lgb.fit(X_train, y_train)
models['LightGBM'] = lgb

xgb_m = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, use_label_encoder=False, eval_metric='logloss')
xgb_m.fit(X_train, y_train)
models['XGBoost'] = xgb_m

for name, m in models.items():
    p = m.predict(X_test)
    print(f"{name}: Acc={accuracy_score(y_test, p):.4f}  F1={f1_score(y_test, p, average='weighted'):.4f}")

CatBoost: Acc=0.9233  F1=0.9200
LightGBM: Acc=0.9204  F1=0.9175
XGBoost: Acc=0.9199  F1=0.9172


## Top 3 Model Selection

In [13]:
all_results = {}
for name, m in models.items():
    p = m.predict(X_test)
    all_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }
results_df = pd.DataFrame(all_results).T.sort_values('F1', ascending=False)
print(results_df)
top3_names = results_df.head(3).index.tolist()
print(f'\nTop 3: {top3_names}')

          Accuracy        F1  Precision    Recall
CatBoost  0.923282  0.920034   0.918294  0.923282
LightGBM  0.920369  0.917543   0.915770  0.920369
XGBoost   0.919883  0.917175   0.915422  0.919883

Top 3: ['CatBoost', 'LightGBM', 'XGBoost']


## Final Evaluation of Top 3

In [14]:
final_results = {}
for name in top3_names:
    m = models[name]
    p = m.predict(X_test)
    final_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }
    print(f"{name}: Acc={final_results[name]['Accuracy']:.4f}  F1={final_results[name]['F1']:.4f}")
    print(classification_report(y_test, p))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = list(final_results.keys())
for i, metric in enumerate(['Accuracy', 'F1']):
    vals = [final_results[n][metric] for n in names]
    axes[i].bar(names, vals, color=['#2ecc71','#3498db','#e74c3c'])
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top3_comparison.png'), dpi=100)
plt.show()

CatBoost: Acc=0.9233  F1=0.9200
              precision    recall  f1-score   support

           0       0.95      0.97      0.96      7310
           1       0.69      0.57      0.63       928

    accuracy                           0.92      8238
   macro avg       0.82      0.77      0.79      8238
weighted avg       0.92      0.92      0.92      8238

LightGBM: Acc=0.9204  F1=0.9175
              precision    recall  f1-score   support

           0       0.95      0.96      0.96      7310
           1       0.67      0.57      0.62       928

    accuracy                           0.92      8238
   macro avg       0.81      0.77      0.79      8238
weighted avg       0.92      0.92      0.92      8238

XGBoost: Acc=0.9199  F1=0.9172
              precision    recall  f1-score   support

           0       0.95      0.96      0.96      7310
           1       0.67      0.57      0.62       928

    accuracy                           0.92      8238
   macro avg       0.81      0.77

## Error Analysis

In [15]:
best_name = top3_names[0]
best_model = models[best_name]
preds = best_model.predict(X_test)

from sklearn.metrics import ConfusionMatrixDisplay
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=axes[0], cmap='Blues')
axes[0].set_title(f'Confusion Matrix ({best_name})')

if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)
    imp.tail(15).plot.barh(ax=axes[1])
    axes[1].set_title('Feature Importance')
else:
    axes[1].text(0.5, 0.5, 'No feature importance', ha='center')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

err = (preds != y_test.values).sum()
print(f'Errors: {err}, Error rate: {err/len(y_test):.4f}')

Errors: 632, Error rate: 0.0767


## Interpretation & Business Insight
Call duration, previous campaign outcome, and economic indicators (euribor3m) strongly predict subscription. Targeting clients with longer engagement and positive previous contact improves conversion.

## Limitations
- Duration is post-hoc (leakage risk)
- Imbalanced (~11% positive)
- Economic context may change

## How to Improve
- Remove duration for realistic production model
- Cost-sensitive learning
- Add customer lifetime value estimation

## Production Considerations
- Score leads before campaign
- Prioritize high-probability clients
- Monitor conversion rate drift

## Common Mistakes
- Using duration as feature in production (leakage)
- Using accuracy on imbalanced data
- Not considering campaign cost per contact

## Mini Challenge
1. Remove 'duration' and retrain — how much does F1 drop?
2. Try class_weight='balanced'
3. Build a lift chart for campaign targeting

## Final Summary
Predicted term deposit subscriptions from marketing data. Boosting models handle the imbalanced response well.

In [16]:
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in final_results.items()}
metrics['best_model'] = top3_names[0]
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics.json')
print(json.dumps(metrics, indent=2))

Saved metrics.json
{
  "CatBoost": {
    "Accuracy": 0.9233,
    "F1": 0.92,
    "Precision": 0.9183,
    "Recall": 0.9233
  },
  "LightGBM": {
    "Accuracy": 0.9204,
    "F1": 0.9175,
    "Precision": 0.9158,
    "Recall": 0.9204
  },
  "XGBoost": {
    "Accuracy": 0.9199,
    "F1": 0.9172,
    "Precision": 0.9154,
    "Recall": 0.9199
  },
  "best_model": "CatBoost"
}
